# NovaMarket, semana 13: ingeniería de características

Partimos del dataset integrado de la semana 12 (ventas, bodega y CRM en una sola tabla) y construimos
sobre él un catálogo de variables nuevas en tres familias: variables derivadas, agregaciones por cliente
y features de dominio de retail.

El dataset integrado tiene una fila por pedido y columnas en bruto (fechas, montos, categorías). Ninguna
de esas columnas responde por sí sola preguntas como "¿este cliente está activo o dormido?" o "¿cuánto
vale este cliente para NovaMarket?". Eso es lo que fabricamos aquí.

Entregables de la semana:

- este notebook, `S13_ingenieria_de_caracteristicas.ipynb`
- el diccionario `S13_diccionario_de_features.md`
- el dataset ampliado `Datos/novamarket_con_features.csv`

## Carga del dataset integrado

In [1]:
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

RUTA_INTEGRADO = "S12_PD_Borja_DatasetIntegrado.csv"

df = pd.read_csv(RUTA_INTEGRADO, low_memory=False)
filas_iniciales, columnas_iniciales = df.shape
df.shape

(620000, 127)

In [2]:
df.head(3)

,id_pedido,id_cliente,fecha_pedido,precio_unitario,unidades_vendidas,monto_compra,edad_cliente,correo_cliente,peso_pedido_kg,fecha_actualizacion_stock,ciudad_tienda_imputada,satisfaccion_sin_dato,correo_sin_dato,sin_comentario,precio_sin_dato,monto_sin_dato,fecha_pedido_sin_dato,fecha_pedido_invalida,precio_unitario_atipico,unidades_vendidas_atipico,edad_cliente_atipico,monto_atipico,peso_pedido_kg_original,peso_capado,precio_unitario_esc,unidades_vendidas_esc,monto_compra_esc,edad_cliente_esc,peso_pedido_kg_esc,codigo_postal_freq,codigo_postal_te,canal_compra_app,canal_compra_marketplace,canal_compra_tienda,canal_compra_web,metodo_pago_efectivo,metodo_pago_paypal,metodo_pago_tarjeta,metodo_pago_transferencia,categoria_producto_belleza,...,producto_lampara_de_escritorio,producto_monitor_24_pulgadas,producto_muneca_articulada,producto_olla_a_presion,producto_parlante_bluetooth,producto_perfume_citrico,producto_pesas_ajustables,producto_rompecabezas_1000_piezas,producto_secador_de_pelo,producto_set_de_bloques,producto_set_de_cuchillos,producto_teclado_mecanico,producto_zapatos_deportivos,nivel_lealtad_ord,nivel_satisfaccion_ord,comentario_cliente,mes_pedido,trimestre_pedido,dia_semana_pedido,es_fin_semana_pedido,longitud_comentario,num_palabras_comentario,tiene_comentario,menciona_demora,grupo_edad,monto_compra_log,monto_compra_boxcox,fuga_cliente,conjunto,categoria_producto,ciudad_tienda,proveedor_principal,bodega_asignada,dias_reposicion,stock_minimo,ciudad_residencia,fecha_inscripcion_club,puntos_fidelidad_actuales,segmento_crm,acepta_promociones
0,P505195,C45650,2025-11-15,28700.0,1.0,28700.0,32.0,cliente405195@novamarket.com.co,14.9,2026-04-17,False,False,False,False,False,False,False,False,False,False,False,False,14.9,False,0.015376,0.2,-0.776385,-0.867977,0.864865,0.006280,0.210914,0,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,3.0,0.0,El domiciliario fue muy amable,11.0,4.0,5.0,1.0,30,5,1,0,26 a 35 (adulto joven),10.264687,85.710632,0,prueba,moda,bogota,Textiles Caribe Ltda.,Bodega Norte,21.0,350.0,Ibagué,2025-07-02,1599.0,Nuevo,Si
1,P677836,C84260,2026-03-05,195700.0,2.0,391400.0,73.0,cliente577836@yahoo.es,12.9,2026-06-10,False,False,False,False,False,False,False,False,False,False,False,False,12.9,False,0.204183,0.4,-0.136251,1.581497,0.324324,0.030268,0.210076,0,0,0,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0.0,1.0,El producto llegó dañado,3.0,1.0,3.0,0.0,24,4,1,0,66 o más (adulto mayor),12.877488,206.394178,0,entrenamiento,electronica,manizales,TecnoAndina S.A.S.,Bodega Norte,7.0,120.0,Manizales,2024-06-25,881.0,Inactivo,No
2,P387869,C164166,2025-03-13,597600.0,NaN,NaN,32.0,cliente287869@novamarket.com.co,14.3,2026-02-26,False,False,False,False,False,False,False,False,False,True,False,True,14.3,False,0.658564,NaN,NaN,-0.867977,0.702703,0.029681,0.213490,0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,2.0,0.0,El domiciliario fue muy amable,3.0,1.0,3.0,0.0,30,5,1,0,26 a 35 (adulto joven),NaN,NaN,0,prueba,jugueteria,cucuta,Juguetes Arcoíris,Bodega Centro,30.0,90.0,Bucaramanga,2023-03-07,3195.0,Frecuente,No


## Fecha de referencia del corte de análisis

Todas las variables de tiempo se calculan contra una fecha fija y documentada, no contra la fecha del día
en que se corre el notebook. Si usáramos `pd.Timestamp("today")`, cada ejecución daría valores distintos y
nadie podría reproducir los resultados.

El dataset tiene pedidos entre enero de 2025 y mayo de 2026 de forma continua, y después un bloque
disperso de 16.000 pedidos fechados a lo largo de 2027, sin nada en el medio. El corte se fija el día
siguiente al último pedido del periodo continuo, el 16 de mayo de 2026. Los pedidos de 2027 quedan
entonces del lado futuro del corte: conservan todas sus columnas en el dataset final, pero no entran a
calcular ninguna agregación.

In [3]:
df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"], errors="coerce")
df["fecha_inscripcion_club"] = pd.to_datetime(df["fecha_inscripcion_club"], errors="coerce")

FECHA_CORTE = pd.Timestamp("2026-05-16")
VENTANA_DIAS = 90

pd.Series({
    "primer_pedido": df["fecha_pedido"].min(),
    "ultimo_pedido": df["fecha_pedido"].max(),
    "ultimo_pedido_antes_del_corte": df.loc[df["fecha_pedido"] < FECHA_CORTE, "fecha_pedido"].max(),
    "fecha_corte": FECHA_CORTE,
    "inicio_ventana_reciente": FECHA_CORTE - pd.Timedelta(days=VENTANA_DIAS),
    "pedidos_despues_del_corte": int((df["fecha_pedido"] >= FECHA_CORTE).sum()),
    "pedidos_sin_fecha": int(df["fecha_pedido"].isna().sum()),
}).to_frame("valor")

,valor
primer_pedido,2025-01-01 00:00:00
ultimo_pedido,2027-12-28 00:00:00
ultimo_pedido_antes_del_corte,2026-05-15 00:00:00
fecha_corte,2026-05-16 00:00:00
inicio_ventana_reciente,2026-02-15 00:00:00
pedidos_despues_del_corte,16000
pedidos_sin_fecha,22000


## Ingeniería de características

### Base de agregación: control de fuga de información

Las agregaciones por cliente no se calculan sobre todo el dataset. Se calculan sobre una base restringida
por dos condiciones:

1. solo pedidos del conjunto de entrenamiento, para que ninguna estadística del conjunto de prueba entre
   en una variable que después alimenta el modelo
2. solo pedidos anteriores a la fecha de corte, para que ninguna feature use información posterior al
   momento que se está analizando

Los pedidos sin fecha registrada quedan fuera de la base porque no se puede ubicar en el tiempo.

In [4]:
COLS_CANAL = ["canal_compra_app", "canal_compra_marketplace", "canal_compra_tienda", "canal_compra_web"]

es_historia = (
    (df["conjunto"] == "entrenamiento")
    & df["fecha_pedido"].notna()
    & (df["fecha_pedido"] < FECHA_CORTE)
)

base = df.loc[
    es_historia,
    ["id_cliente", "id_pedido", "fecha_pedido", "monto_compra", "categoria_producto",
     "puntos_fidelidad_actuales"] + COLS_CANAL,
].copy()

pd.Series({
    "filas_del_dataset": len(df),
    "filas_en_la_base": len(base),
    "clientes_del_dataset": df["id_cliente"].nunique(),
    "clientes_en_la_base": base["id_cliente"].nunique(),
    "clientes_sin_historial_en_la_base": df["id_cliente"].nunique() - base["id_cliente"].nunique(),
}).to_frame("valor")

,valor
filas_del_dataset,620000
filas_en_la_base,465665
clientes_del_dataset,152176
clientes_en_la_base,147353
clientes_sin_historial_en_la_base,4823


### Variables derivadas a nivel de pedido

Cuatro variables nuevas que se calculan con una fórmula sobre una o dos columnas de la misma fila.

- `dia_del_mes_pedido` = día calendario extraído de `fecha_pedido`
- `es_quincena_pago` = verdadero si `dia_del_mes_pedido` cae en 14, 15, 16, 29, 30 o 31
- `es_miembro_club_al_corte` = verdadero si el cliente ya estaba inscrito al club del CRM a la fecha de corte
- `antiguedad_club_dias` = `FECHA_CORTE` - `fecha_inscripcion_club`, en días, solo para quienes ya eran
  miembros a la fecha de corte

In [5]:
df["dia_del_mes_pedido"] = df["fecha_pedido"].dt.day.astype("Int64")

DIAS_DE_PAGO = [14, 15, 16, 29, 30, 31]
df["es_quincena_pago"] = df["dia_del_mes_pedido"].isin(DIAS_DE_PAGO).astype("boolean")
df.loc[df["dia_del_mes_pedido"].isna(), "es_quincena_pago"] = pd.NA

df[["dia_del_mes_pedido", "es_quincena_pago"]].head()

,dia_del_mes_pedido,es_quincena_pago
0,15,True
1,5,False
2,13,False
3,27,False
4,1,False


In [6]:
df["dia_del_mes_pedido"].describe().to_frame("dia_del_mes_pedido")

,dia_del_mes_pedido
count,598000.0
mean,15.427758
std,8.754332
min,1.0
25%,8.0
50%,15.0
75%,23.0
max,31.0


In [7]:
df["es_quincena_pago"].value_counts(dropna=False).to_frame("conteo")

,conteo
es_quincena_pago,
False,494676
True,103324
<NA>,22000


Un candidato más para esta familia era el ticket relativo a la categoría: el monto del pedido dividido por
el monto promedio de su categoría, con la idea de que 800.000 pesos no significan lo mismo en electrónica
que en papelería. Antes de agregarlo hay que mirar el denominador.

In [8]:
monto_promedio_categoria = base.groupby("categoria_producto")["monto_compra"].mean()

pd.Series({
    "promedio_mas_bajo": round(float(monto_promedio_categoria.min()), 0),
    "promedio_mas_alto": round(float(monto_promedio_categoria.max()), 0),
    "diferencia_relativa_pct": round(
        float(monto_promedio_categoria.max() / monto_promedio_categoria.min() - 1) * 100, 2
    ),
}).to_frame("monto_promedio_por_categoria")

,monto_promedio_por_categoria
promedio_mas_bajo,650291.00
promedio_mas_alto,656531.00
diferencia_relativa_pct,0.96


Las seis categorías de NovaMarket tienen montos promedio que difieren menos del 1 % entre sí. Dividir por
ese promedio equivale a dividir por una constante, así que la variable resultante sería el mismo
`monto_compra` en otra escala y no aportaría ninguna señal nueva. Ese candidato queda descartado. La
comparación que sí aporta información es contra el gasto habitual del propio cliente, y esa necesita las
agregaciones, así que se calcula más abajo.

La inscripción al club del CRM llega hasta diciembre de 2026, es decir, después de la fecha de corte. Un
cliente inscrito después del corte todavía no era miembro en el momento que se está analizando, así que su
antigüedad no es cero ni negativa: no existe. Por eso la antigüedad se calcula solo para quienes ya eran
miembros, y la condición queda guardada aparte en una bandera.

In [9]:
df["es_miembro_club_al_corte"] = (
    df["fecha_inscripcion_club"].notna() & (df["fecha_inscripcion_club"] < FECHA_CORTE)
)

df["antiguedad_club_dias"] = np.where(
    df["es_miembro_club_al_corte"],
    (FECHA_CORTE - df["fecha_inscripcion_club"]).dt.days,
    np.nan,
)

df["es_miembro_club_al_corte"].value_counts(dropna=False).to_frame("pedidos")

,pedidos
es_miembro_club_al_corte,
True,347984
False,272016


In [10]:
df["antiguedad_club_dias"].describe().round(1).to_frame("antiguedad_club_dias")

,antiguedad_club_dias
count,347984.0
mean,616.9
std,356.3
min,1.0
25%,309.0
50%,616.0
75%,925.0
max,1231.0


Lectura de las verificaciones anteriores:

- `dia_del_mes_pedido` va de 1 a 31, que es el rango esperado de un día calendario
- `antiguedad_club_dias` arranca en 1 día y no tiene valores negativos porque se restringe a los clientes
  que ya eran miembros a la fecha de corte. Sus nulos son de dos tipos: los pedidos cuyo cliente no apareció
  en el CRM, el mismo grupo de la semana 12, y los pedidos de clientes que se inscribieron después del corte

### Agregaciones por cliente

El dataset tiene una fila por pedido, pero las preguntas de Camila son por cliente. Estas cinco variables
resumen todos los pedidos de la base de un mismo `id_cliente` en un solo número.

- `numero_compras_cliente` = conteo de pedidos del cliente en la base
- `total_gastado_cliente` = suma de `monto_compra` del cliente en la base
- `gasto_promedio_cliente` = `total_gastado_cliente` / pedidos del cliente con monto registrado
- `categorias_distintas_cliente` = número de categorías distintas que compró el cliente
- `compras_ultimos_90_dias` = conteo de pedidos del cliente en los 90 días previos a la fecha de corte

`fecha_ultima_compra` y `puntos_fidelidad_cliente` se calculan aquí como insumos de las variables de las dos
secciones siguientes, y no se suman al dataset final.

In [11]:
resumen = base.groupby("id_cliente").agg(
    numero_compras_cliente=("id_pedido", "count"),
    pedidos_con_monto=("monto_compra", "count"),
    total_gastado_cliente=("monto_compra", "sum"),
    gasto_promedio_cliente=("monto_compra", "mean"),
    categorias_distintas_cliente=("categoria_producto", "nunique"),
    fecha_ultima_compra=("fecha_pedido", "max"),
    puntos_fidelidad_cliente=("puntos_fidelidad_actuales", "max"),
).reset_index()

resumen.loc[resumen["pedidos_con_monto"] == 0, "total_gastado_cliente"] = np.nan

resumen.head()

,id_cliente,numero_compras_cliente,pedidos_con_monto,total_gastado_cliente,gasto_promedio_cliente,categorias_distintas_cliente,fecha_ultima_compra,puntos_fidelidad_cliente
0,C10000,6,6,2768400.0,461400.0,4,2025-09-13,11414.0
1,C100000,7,6,2393400.0,398900.0,3,2026-02-14,11891.0
2,C100001,3,3,1306500.0,435500.0,2,2025-12-01,11667.0
3,C100002,6,5,2878200.0,575640.0,5,2026-04-05,10905.0
4,C100003,2,2,3057400.0,1528700.0,2,2025-07-05,4040.0


Un cliente cuyos pedidos no tienen ningún monto registrado queda con `total_gastado_cliente` nulo en lugar
de cero: no gastó cero, no sabemos cuánto gastó. La suma de pandas devuelve cero en ese caso, por eso se
corrige de forma explícita con `pedidos_con_monto`.

In [12]:
inicio_ventana = FECHA_CORTE - pd.Timedelta(days=VENTANA_DIAS)
recientes = base.loc[base["fecha_pedido"] >= inicio_ventana]

conteo_ventana = (
    recientes.groupby("id_cliente")["id_pedido"]
    .count()
    .rename("compras_ultimos_90_dias")
)

resumen = resumen.merge(conteo_ventana, on="id_cliente", how="left")
resumen["compras_ultimos_90_dias"] = resumen["compras_ultimos_90_dias"].fillna(0).astype(int)

resumen[["numero_compras_cliente", "total_gastado_cliente", "gasto_promedio_cliente",
         "categorias_distintas_cliente", "compras_ultimos_90_dias"]].describe().round(2)

,numero_compras_cliente,total_gastado_cliente,gasto_promedio_cliente,categorias_distintas_cliente,compras_ultimos_90_dias
count,147353.00,143829.00,143829.00,147353.00,147353.00
mean,3.16,1841498.23,653269.43,2.49,0.57
std,1.63,1401168.70,419832.79,1.10,0.74
min,1.00,0.00,0.00,1.00,0.00
25%,2.00,768800.00,372525.00,2.00,0.00
50%,3.00,1514000.00,560933.33,2.00,0.00
75%,4.00,2570100.00,826950.00,3.00,1.00
max,14.00,13751000.00,4438500.00,6.00,6.00


Lectura de la verificación: el conteo de compras por cliente tiene mediana de 3 sobre un historial de
dieciséis meses. `compras_ultimos_90_dias` arranca en cero porque hay clientes que no compraron nada en el
último trimestre antes del corte, y ese cero es información, no un dato faltante. El gasto promedio no
tiene valores negativos ni infinitos.

### Variables derivadas sobre las agregaciones

Tres variables nuevas que se calculan con una fórmula sobre las agregaciones de la sección anterior.

- `dias_desde_ultima_compra` = `FECHA_CORTE` - `fecha_ultima_compra`, en días
- `es_cliente_recurrente` = verdadero si `numero_compras_cliente` es mayor que 1
- `puntos_por_mil_gastados` = `puntos_fidelidad_cliente` / (`total_gastado_cliente` / 1000)

In [13]:
resumen["dias_desde_ultima_compra"] = (FECHA_CORTE - resumen["fecha_ultima_compra"]).dt.days

resumen["dias_desde_ultima_compra"].describe().round(1).to_frame("dias_desde_ultima_compra")

,dias_desde_ultima_compra
count,147353.0
mean,141.0
std,118.2
min,1.0
25%,46.0
50%,108.0
75%,208.0
max,500.0


In [14]:
resumen["es_cliente_recurrente"] = resumen["numero_compras_cliente"] > 1

resumen["es_cliente_recurrente"].value_counts(dropna=False).to_frame("conteo")

,conteo
es_cliente_recurrente,
True,124102
False,23251


La división de `puntos_por_mil_gastados` se protege contra el cero y el nulo: solo se calcula cuando el
gasto en miles es mayor que cero, y en los demás casos queda nulo. Sin esa condición, un cliente con gasto
cero produciría un valor infinito.

In [15]:
gasto_en_miles = resumen["total_gastado_cliente"] / 1000

resumen["puntos_por_mil_gastados"] = np.where(
    gasto_en_miles > 0,
    resumen["puntos_fidelidad_cliente"] / gasto_en_miles,
    np.nan,
)

pd.Series({
    "nulos": int(resumen["puntos_por_mil_gastados"].isna().sum()),
    "infinitos": int(np.isinf(resumen["puntos_por_mil_gastados"].fillna(0)).sum()),
    "minimo": round(float(resumen["puntos_por_mil_gastados"].min()), 3),
    "maximo": round(float(resumen["puntos_por_mil_gastados"].max()), 3),
    "mediana": round(float(resumen["puntos_por_mil_gastados"].median()), 3),
}).to_frame("puntos_por_mil_gastados")

,puntos_por_mil_gastados
nulos,16159.000
infinitos,0.000
minimo,0.000
maximo,555.802
mediana,4.605


Lectura de las tres verificaciones: la recencia mínima es de un día, nunca negativa, porque la fecha de
corte es posterior al último pedido. La bandera de recurrencia reparte a los clientes entre quienes
compraron una sola vez y quienes volvieron. Los puntos por mil pesos gastados no tienen infinitos y los
nulos son los clientes sin monto registrado o sin registro en el CRM.

### Features de dominio

Cinco variables que no salen de una fórmula matemática genérica, sino del criterio con el que el equipo de
marketing de NovaMarket ya piensa a sus clientes. Todas se apoyan en las variables derivadas y las
agregaciones de las secciones anteriores.

- `rfm_score` = `score_R` + `score_F` + `score_M`, donde cada componente es el cuartil del cliente en
  recencia, frecuencia y monto, en escala de 1 a 4
- `segmento_rfm` = etiqueta de negocio asignada por tramos de `rfm_score`
- `categoria_favorita` = categoría en la que el cliente concentra más pedidos, y si hay empate, la de su
  compra más reciente
- `canal_principal` = canal por el que el cliente compra más veces, con el mismo criterio de desempate
- `es_temporada_alta` = verdadero si el pedido cae en noviembre o diciembre

In [16]:
resumen["score_R"] = pd.qcut(resumen["dias_desde_ultima_compra"], q=4, labels=[4, 3, 2, 1])
resumen["score_F"] = pd.qcut(resumen["numero_compras_cliente"].rank(method="first"), q=4, labels=[1, 2, 3, 4])
resumen["score_M"] = pd.qcut(resumen["total_gastado_cliente"].rank(method="first"), q=4, labels=[1, 2, 3, 4])

COMPONENTES_RFM = ["score_R", "score_F", "score_M"]
con_rfm_completo = resumen[COMPONENTES_RFM].notna().all(axis=1)

resumen["rfm_score"] = np.nan
resumen.loc[con_rfm_completo, "rfm_score"] = (
    resumen.loc[con_rfm_completo, "score_R"].astype(int)
    + resumen.loc[con_rfm_completo, "score_F"].astype(int)
    + resumen.loc[con_rfm_completo, "score_M"].astype(int)
)

resumen["rfm_score"].value_counts(dropna=False).sort_index().to_frame("clientes")

,clientes
rfm_score,
3.0,10339
4.0,11475
5.0,13993
6.0,16426
7.0,17390
8.0,18068
9.0,17402
10.0,16459
11.0,13597


En recencia la escala va al revés que en las otras dos: menos días desde la última compra es mejor, por eso
las etiquetas de `score_R` son 4, 3, 2, 1. En frecuencia y monto se aplica `rank(method="first")` antes de
cortar en cuartiles porque hay muchos clientes empatados en el mismo número de compras y `qcut` no puede
partir una columna con bordes repetidos.

In [17]:
resumen["segmento_rfm"] = pd.cut(
    resumen["rfm_score"],
    bins=[2, 4, 6, 8, 10, 12],
    labels=["Perdido", "En riesgo", "Regular", "Leal", "Campeon"],
)

resumen["segmento_rfm"].value_counts(dropna=False).to_frame("clientes")

,clientes
segmento_rfm,
Regular,35458
Leal,33861
En riesgo,30419
Campeon,22277
Perdido,21814
NaN,3524


In [18]:
def reconstruir_desde_indicadoras(datos, columnas, prefijo):
    """Devuelve la etiqueta original a partir de un bloque de columnas indicadoras."""
    etiquetas = [c[len(prefijo):] for c in columnas]
    posicion = datos[columnas].to_numpy().argmax(axis=1)
    return pd.Series([etiquetas[i] for i in posicion], index=datos.index)


base["canal_compra"] = reconstruir_desde_indicadoras(base, COLS_CANAL, "canal_compra_")

base["canal_compra"].value_counts().to_frame("pedidos")

,pedidos
canal_compra,
tienda,116670
web,116592
marketplace,116431
app,115972


El canal de compra quedó convertido en cuatro columnas indicadoras en la semana 9. Para contar cuál usa más
cada cliente hay que recuperar la etiqueta original, y eso se puede hacer sin pérdida porque cada fila tiene
exactamente un uno entre las cuatro columnas.

El cliente promedio tiene 3 pedidos repartidos en 2,5 categorías, así que los empates son la norma: lo más
común es que dos categorías tengan un pedido cada una. Tomar el máximo sin más desempata por orden
alfabético, y eso infla a `belleza` de forma artificial. El desempate correcto es de negocio: entre dos
categorías con el mismo número de pedidos, la favorita es la de la compra más reciente.

In [19]:
def etiqueta_dominante(datos, columna, nombre_salida):
    """Etiqueta con mas pedidos por cliente, desempatando por la compra mas reciente."""
    conteo = (
        datos.groupby(["id_cliente", columna])
        .agg(pedidos=("id_pedido", "count"), ultima_compra=("fecha_pedido", "max"))
        .reset_index()
        .sort_values(["id_cliente", "pedidos", "ultima_compra"], ascending=[True, False, False])
    )
    return conteo.groupby("id_cliente")[columna].first().rename(nombre_salida)


resumen = resumen.merge(
    etiqueta_dominante(base, "categoria_producto", "categoria_favorita"), on="id_cliente", how="left"
)
resumen = resumen.merge(
    etiqueta_dominante(base, "canal_compra", "canal_principal"), on="id_cliente", how="left"
)

pd.concat([
    resumen["categoria_favorita"].value_counts(dropna=False).rename("categoria_favorita"),
    resumen["canal_principal"].value_counts(dropna=False).rename("canal_principal"),
], axis=1)

,categoria_favorita,canal_principal
belleza,24863.0,NaN
electronica,24805.0,NaN
deportes,24571.0,NaN
hogar,24526.0,NaN
jugueteria,24387.0,NaN
moda,24201.0,NaN
tienda,NaN,37236.0
web,NaN,37017.0
marketplace,NaN,36693.0
app,NaN,36407.0


In [20]:
df["es_temporada_alta"] = df["fecha_pedido"].dt.month.isin([11, 12]).astype("boolean")
df.loc[df["fecha_pedido"].isna(), "es_temporada_alta"] = pd.NA

df["es_temporada_alta"].value_counts(dropna=False).to_frame("pedidos")

,pedidos
es_temporada_alta,
False,524001
True,73999
<NA>,22000


Lectura de las verificaciones de dominio: el `rfm_score` se concentra entre 6 y 9, con pocos clientes en los
extremos 3 y 12, que es el comportamiento esperado al sumar tres cuartiles independientes. La categoría
favorita y el canal principal quedan repartidos de forma casi uniforme entre las seis categorías y los
cuatro canales, que es lo que se espera cuando el desempate no privilegia a ninguna etiqueta. La temporada alta cubre dos meses de doce, así que una sexta parte de los pedidos.

## Unión de las features al dataset

El resumen tiene una fila por cliente y el dataset tiene una fila por pedido, así que cada variable de
cliente se pega al dataset buscando el `id_cliente` de la fila. Se hace columna por columna con `map` en
lugar de un `merge`, porque un `merge` construye una copia completa de una tabla de 620.000 filas por 127
columnas y el equipo trabaja en máquinas de 4 GB de memoria. El resultado es el mismo que un
`merge` con `how="left"`: los clientes sin historial quedan en nulo y no se pierde ni se duplica ninguna
fila.

Los componentes intermedios (`pedidos_con_monto`, `fecha_ultima_compra`, `puntos_fidelidad_cliente`,
`score_R`, `score_F`, `score_M`) no se suman al dataset final: no aportan señal propia más allá de las
variables que ya se construyeron con ellos.

La unión deja además una variable de control:

- `sin_historial_cliente` = verdadero si `numero_compras_cliente` quedó nulo, es decir si el cliente no
  tiene ningún pedido en la base de agregación

In [21]:
COLS_RESUMEN = [
    "numero_compras_cliente",
    "total_gastado_cliente",
    "gasto_promedio_cliente",
    "categorias_distintas_cliente",
    "compras_ultimos_90_dias",
    "dias_desde_ultima_compra",
    "es_cliente_recurrente",
    "puntos_por_mil_gastados",
    "rfm_score",
    "segmento_rfm",
    "categoria_favorita",
    "canal_principal",
]

filas_antes_union = df.shape[0]
resumen_por_cliente = resumen.set_index("id_cliente")

for columna in COLS_RESUMEN:
    df[columna] = df["id_cliente"].map(resumen_por_cliente[columna])

df["es_cliente_recurrente"] = df["es_cliente_recurrente"].astype("boolean")
df["sin_historial_cliente"] = df["numero_compras_cliente"].isna()

pd.Series({
    "filas_antes_de_la_union": filas_antes_union,
    "filas_despues_de_la_union": df.shape[0],
    "se_conservan_las_filas": filas_antes_union == df.shape[0],
    "columnas_al_inicio": columnas_iniciales,
    "columnas_al_final": df.shape[1],
}).to_frame("valor")

,valor
filas_antes_de_la_union,620000
filas_despues_de_la_union,620000
se_conservan_las_filas,True
columnas_al_inicio,127
columnas_al_final,145


`sin_historial_cliente` marca de forma explícita a los clientes que no tienen ningún pedido en la base de
entrenamiento anterior al corte. Son los clientes cuyas variables agregadas quedan nulas, y conviene que
eso sea una columna visible en lugar de un hueco sin explicación.

In [22]:
df["sin_historial_cliente"].value_counts(dropna=False).to_frame("pedidos")

,pedidos
sin_historial_cliente,
False,612384
True,7616


### Variable derivada sobre el dataset unido

Ahora que cada pedido tiene al lado el gasto promedio de su cliente, se puede calcular la comparación que
quedó pendiente arriba.

- `ticket_relativo_cliente` = `monto_compra` / `gasto_promedio_cliente`

Un valor de 2 significa que el cliente gastó en ese pedido el doble de lo que suele gastar. La división se
protege contra el cero y el nulo igual que en `puntos_por_mil_gastados`.

In [23]:
df["ticket_relativo_cliente"] = np.where(
    df["gasto_promedio_cliente"] > 0,
    df["monto_compra"] / df["gasto_promedio_cliente"],
    np.nan,
)

df["ticket_relativo_cliente"].describe().round(3).to_frame("ticket_relativo_cliente")

,ticket_relativo_cliente
count,530024.000
mean,1.119
std,1.306
min,0.000
25%,0.488
50%,0.930
75%,1.409
max,126.948


Verificación de que la variable aporta una señal distinta y no es el monto en otra escala: se compara su
correlación con `monto_compra` contra la que habría tenido el candidato descartado.

In [24]:
ticket_por_categoria = df["monto_compra"] / df["categoria_producto"].map(monto_promedio_categoria)

pd.Series({
    "candidato descartado, contra monto_compra": round(
        float(df["monto_compra"].corr(ticket_por_categoria)), 4
    ),
    "ticket_relativo_cliente, contra monto_compra": round(
        float(df["monto_compra"].corr(df["ticket_relativo_cliente"])), 4
    ),
}).to_frame("correlacion")

,correlacion
"candidato descartado, contra monto_compra",1.0000
"ticket_relativo_cliente, contra monto_compra",0.5138


## Verificación final del catálogo

Revisión de las 19 columnas nuevas en bloque: tipo de dato, nulos, infinitos y rango. El criterio no es que
no haya nulos, sino que cada nulo tenga una causa identificada.

In [25]:
FEATURES_NUEVAS = [
    "dia_del_mes_pedido",
    "es_quincena_pago",
    "es_miembro_club_al_corte",
    "antiguedad_club_dias",
    "numero_compras_cliente",
    "total_gastado_cliente",
    "gasto_promedio_cliente",
    "categorias_distintas_cliente",
    "compras_ultimos_90_dias",
    "dias_desde_ultima_compra",
    "es_cliente_recurrente",
    "puntos_por_mil_gastados",
    "rfm_score",
    "segmento_rfm",
    "categoria_favorita",
    "canal_principal",
    "es_temporada_alta",
    "sin_historial_cliente",
    "ticket_relativo_cliente",
]

len(FEATURES_NUEVAS)

19

In [26]:
def contar_infinitos(serie):
    if pd.api.types.is_numeric_dtype(serie) and not pd.api.types.is_bool_dtype(serie):
        return int(np.isinf(serie.to_numpy(dtype="float64", na_value=0.0)).sum())
    return 0


auditoria = pd.DataFrame({
    "tipo": [str(df[c].dtype) for c in FEATURES_NUEVAS],
    "nulos": [int(df[c].isna().sum()) for c in FEATURES_NUEVAS],
    "pct_nulos": [round(float(df[c].isna().mean() * 100), 2) for c in FEATURES_NUEVAS],
    "infinitos": [contar_infinitos(df[c]) for c in FEATURES_NUEVAS],
    "valores_distintos": [int(df[c].nunique(dropna=True)) for c in FEATURES_NUEVAS],
}, index=FEATURES_NUEVAS)

auditoria

,tipo,nulos,pct_nulos,infinitos,valores_distintos
dia_del_mes_pedido,Int64,22000,3.55,0,31
es_quincena_pago,boolean,22000,3.55,0,2
es_miembro_club_al_corte,bool,0,0.00,0,2
antiguedad_club_dias,float64,272016,43.87,0,1135
numero_compras_cliente,float64,7616,1.23,0,14
total_gastado_cliente,float64,15492,2.50,0,45442
gasto_promedio_cliente,float64,15492,2.50,0,62376
categorias_distintas_cliente,float64,7616,1.23,0,6
compras_ultimos_90_dias,float64,7616,1.23,0,7
dias_desde_ultima_compra,float64,7616,1.23,0,500


Las tres causas de nulo en la tabla anterior:

- los pedidos sin fecha registrada afectan a `dia_del_mes_pedido`, `es_quincena_pago` y `es_temporada_alta`
- los pedidos sin monto registrado afectan a `ticket_relativo_cliente`
- los pedidos cuyo cliente no apareció en el CRM, más los clientes inscritos al club después del corte,
  afectan a `antiguedad_club_dias` y a `puntos_por_mil_gastados`
- los clientes sin ningún pedido en la base de entrenamiento anterior al corte afectan a todas las
  variables agregadas y a las que se derivan de ellas, y están marcados en `sin_historial_cliente`

Ninguna columna nueva tiene valores infinitos.

In [27]:
columnas_originales = pd.read_csv(RUTA_INTEGRADO, nrows=0).columns.tolist()
faltantes = [c for c in columnas_originales if c not in df.columns]

pd.Series({
    "columnas_originales": len(columnas_originales),
    "columnas_originales_que_faltan": len(faltantes),
    "features_nuevas": len(FEATURES_NUEVAS),
    "columnas_totales": df.shape[1],
    "filas_iniciales": filas_iniciales,
    "filas_finales": df.shape[0],
    "se_conservan_las_filas": filas_iniciales == df.shape[0],
}).to_frame("valor")

,valor
columnas_originales,127
columnas_originales_que_faltan,0
features_nuevas,19
columnas_totales,146
filas_iniciales,620000
filas_finales,620000
se_conservan_las_filas,True


## Exportación del dataset con features

In [28]:
os.makedirs("Datos", exist_ok=True)
RUTA_SALIDA = "Datos/novamarket_con_features.csv"

df.to_csv(RUTA_SALIDA, index=False, chunksize=50_000)

pd.Series({
    "archivo": RUTA_SALIDA,
    "tamano_mb": round(os.path.getsize(RUTA_SALIDA) / 1024 ** 2, 1),
}).to_frame("valor")

,valor
archivo,Datos/novamarket_con_features.csv
tamano_mb,454.6


In [29]:
verificacion = pd.read_csv(RUTA_SALIDA, nrows=5)

pd.Series({
    "columnas_en_el_archivo": verificacion.shape[1],
    "features_nuevas_presentes": sum(c in verificacion.columns for c in FEATURES_NUEVAS),
}).to_frame("valor")

,valor
columnas_en_el_archivo,146
features_nuevas_presentes,19


## Qué sigue

El catálogo de esta semana queda en 19 variables candidatas. En la semana 14 el trabajo es el inverso:
medir cuáles de estas candidatas aportan señal de verdad al problema de retención y descartar las demás.
La fórmula y la justificación de negocio de cada una quedan en `S13_diccionario_de_features.md`.